# Train/Dev/Test split

In [6]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [7]:
words = open("../data/shakespeare_100k.txt").read().splitlines()


In [8]:
lookup=sorted(list(set(''.join(words))))
stoi={s:i+1 for i,s in enumerate(lookup)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}

In [9]:

window_size = 3
def build_dataset(words):
    X, Y =[], []

    for w in words: 
        context = [0]*window_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:]+[ix] # crop and append

    X=torch.tensor(X)
    Y=torch.tensor(Y)

    return X, Y
import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xde, Ydev = build_dataset(words[n1:n2])
Xts, Yts = build_dataset(words[n2:])


In [10]:
vocab_size = len(stoi)
embedding_dim = 3

C = torch.randn((vocab_size, embedding_dim))
W1 = torch.randn((window_size * embedding_dim, 300))
b1 = torch.randn(300)

W2 = torch.randn((300, vocab_size))
b2 = torch.randn(vocab_size)

parameters = [C, W1, b1, W2, b2]

In [ ]:

# torch.randint(0, X.shape[0], (50,)) # creates random indexes from our data set with batch size 50

for p in parameters:
    p.requires_grad=True

for i in range(1000):
    # forward pass
    emb = C[Xtr] 
    h=torch.tanh(emb.view(-1, 9) @ W1+b1)
    logits = h @ W2 + b2 
    loss = F.cross_entropy(logits, Ytr)
    print(loss)
   
    # backward pass

    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -0.1 * p.grad
print(loss.item())

tensor(2.5764, grad_fn=<NllLossBackward0>)
tensor(2.5664, grad_fn=<NllLossBackward0>)
tensor(2.5757, grad_fn=<NllLossBackward0>)
tensor(2.5657, grad_fn=<NllLossBackward0>)
tensor(2.5751, grad_fn=<NllLossBackward0>)
tensor(2.5650, grad_fn=<NllLossBackward0>)
tensor(2.5744, grad_fn=<NllLossBackward0>)
tensor(2.5643, grad_fn=<NllLossBackward0>)
tensor(2.5737, grad_fn=<NllLossBackward0>)
tensor(2.5637, grad_fn=<NllLossBackward0>)
tensor(2.5731, grad_fn=<NllLossBackward0>)
tensor(2.5630, grad_fn=<NllLossBackward0>)
tensor(2.5725, grad_fn=<NllLossBackward0>)
tensor(2.5624, grad_fn=<NllLossBackward0>)
tensor(2.5719, grad_fn=<NllLossBackward0>)
tensor(2.5618, grad_fn=<NllLossBackward0>)
tensor(2.5713, grad_fn=<NllLossBackward0>)
tensor(2.5612, grad_fn=<NllLossBackward0>)
tensor(2.5707, grad_fn=<NllLossBackward0>)
tensor(2.5606, grad_fn=<NllLossBackward0>)
tensor(2.5701, grad_fn=<NllLossBackward0>)
tensor(2.5601, grad_fn=<NllLossBackward0>)
tensor(2.5696, grad_fn=<NllLossBackward0>)
tensor(2.55

In [33]:

emb = C[Xde] 
h=torch.tanh(emb.view(-1, 9) @ W1+b1)
logits = h @ W2 + b2 
loss = F.cross_entropy(logits, Ydev)
loss #dev loss

tensor(2.5005, grad_fn=<NllLossBackward0>)

In [36]:
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * window_size # initialize with all ...
    while True:
      emb = C[torch.tensor([context])] # (1,block_size,d)
      h = torch.tanh(emb.view(1, -1) @ W1 + b1)
      logits = h @ W2 + b2
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))

Coeettdat.
Courhh.
tun.
I.
and.
ill.
wriod.
Opoogent.
sCoure.
wifl.
or.
ti.
shed.
inm.
beale.
ar.
toweten.
flons.
oueustdin.
.


# 3 layer architecture

In [39]:
vocab_size = len(stoi)
embedding_dim = 3

C = torch.randn((vocab_size, embedding_dim))
W1 = torch.randn((window_size * embedding_dim, 300))
b1 = torch.randn(300)

W2 = torch.randn((300, 100))
b2 = torch.randn(100)

W3 = torch.randn((100, vocab_size))
b3 = torch.randn(vocab_size)

parameters = [C, W1, b1, W2, b2, W3, b3]

In [40]:
for p in parameters:
    p.requires_grad=True

for i in range(10):
    # forward pass
    emb = C[Xtr] 
    h1=torch.tanh(emb.view(-1, 9) @ W1+b1)
    h2 = torch.relu(h1 @ W2+b2)
    logits = h2 @ W3 + b3 
    loss = F.cross_entropy(logits, Ytr)
    print(loss)
   
    # backward pass

    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -0.1 * p.grad
print(loss.item())

tensor(239.0418, grad_fn=<NllLossBackward0>)
tensor(258.5880, grad_fn=<NllLossBackward0>)
tensor(175.3942, grad_fn=<NllLossBackward0>)
tensor(133.0562, grad_fn=<NllLossBackward0>)
tensor(119.6590, grad_fn=<NllLossBackward0>)
tensor(126.3653, grad_fn=<NllLossBackward0>)
tensor(122.9369, grad_fn=<NllLossBackward0>)
tensor(67.5740, grad_fn=<NllLossBackward0>)
tensor(56.1387, grad_fn=<NllLossBackward0>)
tensor(45.1801, grad_fn=<NllLossBackward0>)
45.18012237548828
